In [ ]:
# ============================================================================
# 🚨 脑影像数据泄露诊断与修复完整分析框架
# 
# 目标：
# 1. 严格按受试者重新分割数据
# 2. 建立新的baseline性能（预期会显著下降）
# 3. 量化泄露对性能的影响
# 4. 生成详细的诊断报告和可视化
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import os
import time
import json
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score, 
                           confusion_matrix, classification_report, balanced_accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# 设置matplotlib中文显示（图表标题用英文，日志用中文）
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (15, 10)
plt.style.use('seaborn-v0_8')

class DataLeakageAnalyzer:
    """
    数据泄露诊断与修复分析器
    
    核心功能：
    1. 诊断当前数据分割中的受试者泄露程度
    2. 重新按受试者严格分割数据
    3. 对比泄露前后的性能差异
    4. 生成详细的分析报告
    """
    
    def __init__(self, data_path, save_path='./data_leakage_analysis/'):
        self.data_path = data_path
        self.save_path = save_path
        self.setup_directories()
        
        # 分析结果存储
        self.results = {
            'original_split_analysis': {},
            'subject_level_split_analysis': {},
            'leakage_impact_analysis': {},
            'performance_comparison': {}
        }
        
        print("🚨 数据泄露诊断与修复分析器初始化完成")
        print(f"📁 结果保存路径: {save_path}")
        
    def setup_directories(self):
        """创建保存目录结构"""
        os.makedirs(self.save_path, exist_ok=True)
        os.makedirs(os.path.join(self.save_path, 'visualizations'), exist_ok=True)
        os.makedirs(os.path.join(self.save_path, 'models'), exist_ok=True)
        os.makedirs(os.path.join(self.save_path, 'reports'), exist_ok=True)
        
    def load_data_with_metadata(self):
        """
        加载数据并提取受试者ID和年龄信息
        """
        print("\n" + "="*80)
        print("📂 Phase 1: 数据加载与元数据提取")
        print("="*80)
        
        print("🔄 正在加载TRAIN38.mat数据...")
        
        try:
            with h5py.File(self.data_path, 'r') as f:
                # 加载主要数据
                data = np.array(f['data']).transpose()
                region = np.array(f['region']).transpose()
                prob_idx = np.array(f['prob_idx']).transpose().flatten().astype(int)
                all_age = np.array(f['all_age']).transpose().flatten()
                
                print(f"✅ 数据加载成功")
                print(f"  - 数据矩阵: {data.shape}")
                print(f"  - 标签矩阵: {region.shape}")
                print(f"  - 受试者ID: {prob_idx.shape}, 范围: [{prob_idx.min()}, {prob_idx.max()}]")
                print(f"  - 年龄信息: {all_age.shape}, 范围: [{all_age.min():.1f}, {all_age.max():.1f}]")
                
        except Exception as e:
            print(f"❌ 数据加载失败: {e}")
            raise
        
        # 数据预处理
        print("\n🔧 数据预处理...")
        
        # 处理标签
        if len(region.shape) > 1 and region.shape[1] > 1:
            # one-hot转换为类别索引
            y_labels = np.argmax(region, axis=1)
        else:
            y_labels = region.flatten().astype(int)
        
        # 存储原始数据
        self.original_data = {
            'X': data,
            'y': y_labels,
            'y_onehot': region,
            'subject_ids': prob_idx,
            'ages': all_age,
            'n_subjects': len(np.unique(prob_idx)),
            'n_classes': len(np.unique(y_labels)),
            'n_samples': len(data)
        }
        
        print(f"✅ 数据预处理完成")
        print(f"  - 受试者数量: {self.original_data['n_subjects']}")
        print(f"  - 类别数量: {self.original_data['n_classes']}")
        print(f"  - 总样本数: {self.original_data['n_samples']:,}")
        
        # 受试者信息统计
        print("\n📊 受试者信息统计:")
        subject_stats = []
        unique_subjects = np.unique(prob_idx)
        
        for subject_id in unique_subjects:
            mask = prob_idx == subject_id
            n_samples = np.sum(mask)
            age = all_age[mask][0]  # 所有样本的年龄应该相同
            
            subject_stats.append({
                'subject_id': subject_id,
                'n_samples': n_samples,
                'age': age,
                'percentage': n_samples / len(data) * 100
            })
        
        self.subject_stats_df = pd.DataFrame(subject_stats)
        print(f"  - 平均每受试者样本数: {self.subject_stats_df['n_samples'].mean():.0f}")
        print(f"  - 样本数范围: [{self.subject_stats_df['n_samples'].min():,}, {self.subject_stats_df['n_samples'].max():,}]")
        print(f"  - 年龄范围: [{self.subject_stats_df['age'].min():.1f}, {self.subject_stats_df['age'].max():.1f}] 岁")
        
        return self.original_data
    
    def analyze_original_split_leakage(self, X_train_current, X_val_current, y_train_current, y_val_current):
        """
        分析当前训练/验证分割中的数据泄露程度
        """
        print("\n" + "="*80)
        print("🔍 Phase 2: 当前数据分割泄露诊断")
        print("="*80)
        
        print("🔄 正在分析当前训练/验证分割的受试者分布...")
        
        # 通过特征匹配找到原始数据中对应的受试者ID
        train_subject_ids = self._map_samples_to_subjects(X_train_current)
        val_subject_ids = self._map_samples_to_subjects(X_val_current)
        
        print(f"✅ 受试者映射完成")
        print(f"  - 训练集样本数: {len(X_train_current):,}")
        print(f"  - 验证集样本数: {len(X_val_current):,}")
        
        # 分析受试者重叠
        train_subjects_unique = set(train_subject_ids)
        val_subjects_unique = set(val_subject_ids)
        overlapping_subjects = train_subjects_unique.intersection(val_subjects_unique)
        
        # 计算泄露指标
        leakage_metrics = {
            'train_subjects': len(train_subjects_unique),
            'val_subjects': len(val_subjects_unique),
            'overlapping_subjects': len(overlapping_subjects),
            'overlap_ratio': len(overlapping_subjects) / len(train_subjects_unique) if len(train_subjects_unique) > 0 else 0,
            'train_samples_from_overlapping': sum(1 for sid in train_subject_ids if sid in overlapping_subjects),
            'val_samples_from_overlapping': sum(1 for sid in val_subject_ids if sid in overlapping_subjects),
        }
        
        # 计算泄露严重程度
        leakage_severity = self._calculate_leakage_severity(leakage_metrics, train_subject_ids, val_subject_ids)
        
        self.results['original_split_analysis'] = {
            'leakage_metrics': leakage_metrics,
            'leakage_severity': leakage_severity,
            'train_subject_ids': train_subject_ids,
            'val_subject_ids': val_subject_ids,
            'overlapping_subjects': list(overlapping_subjects)
        }
        
        print(f"\n🚨 数据泄露诊断结果:")
        print(f"  - 训练集涉及受试者: {leakage_metrics['train_subjects']} 个")
        print(f"  - 验证集涉及受试者: {leakage_metrics['val_subjects']} 个")
        print(f"  - 重叠受试者数量: {leakage_metrics['overlapping_subjects']} 个")
        print(f"  - 受试者重叠率: {leakage_metrics['overlap_ratio']*100:.1f}%")
        print(f"  - 泄露严重程度: {leakage_severity['severity_level']} ({leakage_severity['severity_score']:.3f})")
        
        if leakage_metrics['overlap_ratio'] > 0.5:
            print("  ⚠️  严重数据泄露！超过50%的受试者在训练和验证集中重叠")
        elif leakage_metrics['overlap_ratio'] > 0.1:
            print("  ⚠️  中度数据泄露，需要重新分割数据")
        else:
            print("  ✅ 数据泄露程度较低")
            
        return self.results['original_split_analysis']
    
    def _map_samples_to_subjects(self, X_samples):
        """
        通过特征匹配将样本映射回原始受试者ID
        """
        print("🔄 正在进行样本到受试者的映射...")
        
        # 标准化处理
        scaler = StandardScaler()
        X_original_scaled = scaler.fit_transform(self.original_data['X'])
        X_samples_scaled = scaler.transform(X_samples)
        
        mapped_subject_ids = []
        
        # 使用精确匹配
        for i, sample in enumerate(X_samples_scaled):
            # 寻找最接近的原始样本
            distances = np.sum((X_original_scaled - sample) ** 2, axis=1)
            closest_idx = np.argmin(distances)
            
            # 验证匹配度
            if distances[closest_idx] < 1e-10:  # 非常小的阈值，确保精确匹配
                mapped_subject_ids.append(self.original_data['subject_ids'][closest_idx])
            else:
                # 如果找不到精确匹配，使用最近邻的受试者ID
                mapped_subject_ids.append(self.original_data['subject_ids'][closest_idx])
                
        print(f"✅ 样本映射完成，映射了 {len(mapped_subject_ids)} 个样本")
        return mapped_subject_ids
    
    def _calculate_leakage_severity(self, metrics, train_ids, val_ids):
        """
        计算数据泄露的严重程度
        """
        overlap_ratio = metrics['overlap_ratio']
        
        # 计算样本级别的泄露比例
        train_samples_leaked = metrics['train_samples_from_overlapping'] / len(train_ids)
        val_samples_leaked = metrics['val_samples_from_overlapping'] / len(val_ids)
        
        # 综合评分
        severity_score = (overlap_ratio * 0.4 + 
                         train_samples_leaked * 0.3 + 
                         val_samples_leaked * 0.3)
        
        # 确定严重程度等级
        if severity_score > 0.8:
            severity_level = "Severe"
        elif severity_score > 0.5:
            severity_level = "High"
        elif severity_score > 0.2:
            severity_level = "Moderate"
        elif severity_score > 0.05:
            severity_level = "Low"
        else:
            severity_level = "Minimal"
            
        return {
            'severity_score': severity_score,
            'severity_level': severity_level,
            'train_samples_leaked_ratio': train_samples_leaked,
            'val_samples_leaked_ratio': val_samples_leaked
        }
    
    def create_subject_level_splits(self, test_size=0.2, random_state=42):
        """
        创建严格的受试者级别数据分割
        """
        print("\n" + "="*80)
        print("🔄 Phase 3: 创建严格的受试者级别数据分割")
        print("="*80)
        
        # 获取所有受试者
        unique_subjects = np.unique(self.original_data['subject_ids'])
        print(f"📊 总受试者数: {len(unique_subjects)}")
        
        # 分离测试受试者（prob_idx=38）
        test_subject = 38
        train_val_subjects = unique_subjects[unique_subjects != test_subject]
        
        print(f"  - 训练+验证受试者: {len(train_val_subjects)} 个 (ID: {train_val_subjects})")
        print(f"  - 测试受试者: 1 个 (ID: {test_subject})")
        
        # 计算每个受试者的年龄（用于分层）
        subject_ages = {}
        subject_class_dist = {}
        
        for subject_id in train_val_subjects:
            mask = self.original_data['subject_ids'] == subject_id
            subject_ages[subject_id] = self.original_data['ages'][mask][0]
            
            # 计算该受试者的类别分布
            subject_labels = self.original_data['y'][mask]
            class_counts = np.bincount(subject_labels, minlength=self.original_data['n_classes'])
            subject_class_dist[subject_id] = class_counts
        
        # 创建分层变量（基于年龄分组）
        age_groups = self._create_age_groups(subject_ages)
        
        print(f"\n🔄 正在创建分层的受试者级别分割...")
        print(f"  - 年龄分组策略: {len(set(age_groups.values()))} 个年龄组")
        
        # 受试者级别的分层分割
        subjects_list = list(train_val_subjects)
        age_groups_list = [age_groups[sid] for sid in subjects_list]
        
        try:
            train_subjects, val_subjects = train_test_split(
                subjects_list,
                test_size=test_size,
                random_state=random_state,
                stratify=age_groups_list
            )
        except ValueError:
            # 如果分层失败，使用随机分割
            print("  ⚠️ 分层分割失败，使用随机分割")
            train_subjects, val_subjects = train_test_split(
                subjects_list,
                test_size=test_size,
                random_state=random_state
            )
        
        print(f"✅ 受试者分割完成:")
        print(f"  - 训练受试者: {len(train_subjects)} 个")
        print(f"  - 验证受试者: {len(val_subjects)} 个")
        print(f"  - 测试受试者: 1 个")
        
        # 创建样本级别的分割
        splits = self._create_sample_splits(train_subjects, val_subjects, [test_subject])
        
        # 计算分割统计
        split_stats = self._calculate_split_statistics(splits, subject_ages)
        
        self.results['subject_level_split_analysis'] = {
            'splits': splits,
            'split_stats': split_stats,
            'train_subjects': train_subjects,
            'val_subjects': val_subjects,
            'test_subjects': [test_subject],
            'subject_ages': subject_ages
        }
        
        print(f"\n📊 新分割统计:")
        for split_name, stats in split_stats.items():
            print(f"  {split_name}:")
            print(f"    - 样本数: {stats['n_samples']:,}")
            print(f"    - 受试者数: {stats['n_subjects']}")
            print(f"    - 平均年龄: {stats['mean_age']:.1f} ± {stats['std_age']:.1f} 岁")
            print(f"    - 年龄范围: [{stats['age_range'][0]:.1f}, {stats['age_range'][1]:.1f}] 岁")
        
        return splits
    
    def _create_age_groups(self, subject_ages, n_groups=4):
        """
        创建年龄分组用于分层
        """
        ages = list(subject_ages.values())
        age_quantiles = np.quantile(ages, np.linspace(0, 1, n_groups + 1))
        
        age_groups = {}
        for subject_id, age in subject_ages.items():
            group = np.digitize(age, age_quantiles) - 1
            group = max(0, min(n_groups - 1, group))  # 确保在有效范围内
            age_groups[subject_id] = group
            
        return age_groups
    
    def _create_sample_splits(self, train_subjects, val_subjects, test_subjects):
        """
        根据受试者分割创建样本级别的分割
        """
        splits = {}
        
        for split_name, subjects in [('train', train_subjects), ('val', val_subjects), ('test', test_subjects)]:
            # 创建受试者掩码
            subject_mask = np.isin(self.original_data['subject_ids'], subjects)
            
            splits[split_name] = {
                'X': self.original_data['X'][subject_mask],
                'y': self.original_data['y'][subject_mask],
                'y_onehot': self.original_data['y_onehot'][subject_mask],
                'subject_ids': self.original_data['subject_ids'][subject_mask],
                'ages': self.original_data['ages'][subject_mask],
                'subjects': subjects
            }
            
        return splits
    
    def _calculate_split_statistics(self, splits, subject_ages):
        """
        计算每个分割的统计信息
        """
        stats = {}
        
        for split_name, split_data in splits.items():
            subjects = split_data['subjects']
            ages = [subject_ages[sid] for sid in subjects]
            
            stats[split_name] = {
                'n_samples': len(split_data['X']),
                'n_subjects': len(subjects),
                'mean_age': np.mean(ages),
                'std_age': np.std(ages),
                'age_range': [np.min(ages), np.max(ages)],
                'class_distribution': np.bincount(split_data['y'], 
                                                minlength=self.original_data['n_classes'])
            }
            
        return stats
    
    def evaluate_performance_comparison(self, X_train_original, X_val_original, y_train_original, y_val_original):
        """
        对比原始分割和受试者级别分割的性能差异
        """
        print("\n" + "="*80)
        print("⚖️  Phase 4: 性能对比分析")
        print("="*80)
        
        # 获取新的分割数据
        new_splits = self.results['subject_level_split_analysis']['splits']
        
        print("🔄 正在训练对比模型...")
        
        # 标准化数据
        scaler_original = StandardScaler()
        X_train_orig_scaled = scaler_original.fit_transform(X_train_original)
        X_val_orig_scaled = scaler_original.transform(X_val_original)
        
        scaler_new = StandardScaler()
        X_train_new_scaled = scaler_new.fit_transform(new_splits['train']['X'])
        X_val_new_scaled = scaler_new.transform(new_splits['val']['X'])
        X_test_new_scaled = scaler_new.transform(new_splits['test']['X'])
        
        # 训练和评估模型
        results = {}
        
        # 1. 原始分割性能
        print("\n📊 评估原始分割性能...")
        original_performance = self._train_and_evaluate_model(
            X_train_orig_scaled, y_train_original,
            X_val_orig_scaled, y_val_original,
            model_name="Original Split"
        )
        results['original_split'] = original_performance
        
        # 2. 新分割性能（训练-验证）
        print("\n📊 评估新分割性能（训练-验证）...")
        new_split_performance = self._train_and_evaluate_model(
            X_train_new_scaled, new_splits['train']['y'],
            X_val_new_scaled, new_splits['val']['y'],
            model_name="Subject-Level Split (Train-Val)"
        )
        results['subject_level_split'] = new_split_performance
        
        # 3. 新分割性能（训练-测试）
        print("\n📊 评估新分割性能（训练-测试）...")
        new_test_performance = self._train_and_evaluate_model(
            X_train_new_scaled, new_splits['train']['y'],
            X_test_new_scaled, new_splits['test']['y'],
            model_name="Subject-Level Split (Train-Test)"
        )
        results['subject_level_test'] = new_test_performance
        
        # 计算泄露影响
        leakage_impact = self._calculate_leakage_impact(results)
        
        self.results['performance_comparison'] = {
            'model_performances': results,
            'leakage_impact': leakage_impact
        }
        
        print(f"\n📈 性能对比结果:")
        print(f"  原始分割 (带泄露):")
        print(f"    - F1 Score: {original_performance['f1_macro']:.4f}")
        print(f"    - Accuracy: {original_performance['accuracy']:.4f}")
        
        print(f"  受试者级别分割 (训练-验证):")
        print(f"    - F1 Score: {new_split_performance['f1_macro']:.4f}")
        print(f"    - Accuracy: {new_split_performance['accuracy']:.4f}")
        
        print(f"  受试者级别分割 (训练-测试):")
        print(f"    - F1 Score: {new_test_performance['f1_macro']:.4f}")
        print(f"    - Accuracy: {new_test_performance['accuracy']:.4f}")
        
        print(f"\n🚨 数据泄露影响分析:")
        print(f"  - F1性能虚高: {leakage_impact['f1_inflation']*100:.1f}%")
        print(f"  - 准确率虚高: {leakage_impact['accuracy_inflation']*100:.1f}%")
        print(f"  - 预估真实性能: F1={leakage_impact['estimated_true_f1']:.4f}")
        
        return results
    
    def _train_and_evaluate_model(self, X_train, y_train, X_test, y_test, model_name="Model"):
        """
        训练并评估模型性能
        """
        # 使用随机森林作为基准模型
        model = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        
        # 训练模型
        model.fit(X_train, y_train)
        
        # 预测
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)
        
        # 计算指标
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'f1_macro': f1_score(y_test, y_pred, average='macro'),
            'f1_weighted': f1_score(y_test, y_pred, average='weighted'),
            'precision_macro': precision_score(y_test, y_pred, average='macro'),
            'recall_macro': recall_score(y_test, y_pred, average='macro'),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'n_train_samples': len(X_train),
            'n_test_samples': len(X_test),
            'n_classes': len(np.unique(y_test))
        }
        
        print(f"  {model_name}: F1={metrics['f1_macro']:.4f}, Acc={metrics['accuracy']:.4f}")
        
        return metrics
    
    def _calculate_leakage_impact(self, results):
        """
        计算数据泄露对性能的影响
        """
        original_f1 = results['original_split']['f1_macro']
        subject_level_f1 = results['subject_level_split']['f1_macro']
        subject_test_f1 = results['subject_level_test']['f1_macro']
        
        original_acc = results['original_split']['accuracy']
        subject_level_acc = results['subject_level_split']['accuracy']
        subject_test_acc = results['subject_level_test']['accuracy']
        
        # 计算性能虚高程度
        f1_inflation = (original_f1 - subject_level_f1) / subject_level_f1 if subject_level_f1 > 0 else 0
        accuracy_inflation = (original_acc - subject_level_acc) / subject_level_acc if subject_level_acc > 0 else 0
        
        # 预估真实性能（考虑训练-测试差距）
        estimated_true_f1 = min(subject_level_f1, subject_test_f1)
        estimated_true_acc = min(subject_level_acc, subject_test_acc)
        
        return {
            'f1_inflation': f1_inflation,
            'accuracy_inflation': accuracy_inflation,
            'estimated_true_f1': estimated_true_f1,
            'estimated_true_acc': estimated_true_acc,
            'performance_drop_f1': original_f1 - estimated_true_f1,
            'performance_drop_acc': original_acc - estimated_true_acc
        }
    
    def generate_comprehensive_visualizations(self):
        """
        生成完整的可视化分析图表
        """
        print("\n" + "="*80)
        print("📊 Phase 5: 生成综合可视化分析")
        print("="*80)
        
        # 创建大型综合图表
        fig = plt.figure(figsize=(20, 24))
        
        # 1. 受试者分布和年龄信息
        ax1 = plt.subplot(4, 3, 1)
        self._plot_subject_distribution(ax1)
        
        # 2. 数据泄露可视化
        ax2 = plt.subplot(4, 3, 2)
        self._plot_leakage_visualization(ax2)
        
        # 3. 年龄分布对比
        ax3 = plt.subplot(4, 3, 3)
        self._plot_age_distribution(ax3)
        
        # 4. 样本数分布
        ax4 = plt.subplot(4, 3, 4)
        self._plot_sample_distribution(ax4)
        
        # 5. 性能对比条形图
        ax5 = plt.subplot(4, 3, 5)
        self._plot_performance_comparison(ax5)
        
        # 6. 泄露影响量化
        ax6 = plt.subplot(4, 3, 6)
        self._plot_leakage_impact(ax6)
        
        # 7. 类别分布对比
        ax7 = plt.subplot(4, 3, 7)
        self._plot_class_distribution(ax7)
        
        # 8. 性能下降瀑布图
        ax8 = plt.subplot(4, 3, 8)
        self._plot_performance_waterfall(ax8)
        
        # 9. 受试者年龄vs样本数散点图
        ax9 = plt.subplot(4, 3, 9)
        self._plot_age_vs_samples(ax9)
        
        # 10. 训练集规模影响
        ax10 = plt.subplot(4, 3, 10)
        self._plot_training_size_impact(ax10)
        
        # 11. 混淆矩阵对比
        ax11 = plt.subplot(4, 3, 11)
        self._plot_confusion_matrix_comparison(ax11)
        
        # 12. 总结仪表盘
        ax12 = plt.subplot(4, 3, 12)
        self._plot_summary_dashboard(ax12)
        
        plt.tight_layout()
        viz_path = os.path.join(self.save_path, 'visualizations', 'comprehensive_leakage_analysis.png')
        plt.savefig(viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"✅ 综合可视化已保存: {viz_path}")
        
        # 生成单独的高质量图表
        self._generate_individual_plots()
    
    def _plot_subject_distribution(self, ax):
        """绘制受试者分布图"""
        df = self.subject_stats_df
        
        bars = ax.bar(range(len(df)), df['n_samples'], 
                     color='lightblue', alpha=0.7, edgecolor='black')
        
        # 标记测试受试者
        test_idx = df[df['subject_id'] == 38].index
        if len(test_idx) > 0:
            bars[test_idx[0]].set_color('red')
            bars[test_idx[0]].set_alpha(0.8)
        
        ax.set_title('Subject Sample Distribution', fontweight='bold', fontsize=14)
        ax.set_xlabel('Subject Index')
        ax.set_ylabel('Number of Samples')
        ax.grid(True, alpha=0.3)
        
        # 添加平均线
        mean_samples = df['n_samples'].mean()
        ax.axhline(mean_samples, color='red', linestyle='--', alpha=0.7,
                  label=f'Mean: {mean_samples:.0f}')
        ax.legend()
    
    def _plot_leakage_visualization(self, ax):
        """绘制数据泄露可视化"""
        if 'original_split_analysis' not in self.results:
            ax.text(0.5, 0.5, 'Original split analysis not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        metrics = self.results['original_split_analysis']['leakage_metrics']
        
        # 创建Venn图样式的可视化
        labels = ['Train Only', 'Overlap', 'Val Only']
        sizes = [
            metrics['train_subjects'] - metrics['overlapping_subjects'],
            metrics['overlapping_subjects'],
            metrics['val_subjects'] - metrics['overlapping_subjects']
        ]
        colors = ['lightblue', 'red', 'lightgreen']
        
        wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors, 
                                         autopct='%1.1f%%', startangle=90)
        
        ax.set_title('Data Leakage Visualization\n(Subject Overlap)', 
                    fontweight='bold', fontsize=14)
        
        # 添加泄露严重程度标注
        severity = self.results['original_split_analysis']['leakage_severity']
        ax.text(0.02, 0.98, f"Severity: {severity['severity_level']}\nScore: {severity['severity_score']:.3f}", 
               transform=ax.transAxes, va='top', 
               bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.7))
    
    def _plot_age_distribution(self, ax):
        """绘制年龄分布对比"""
        if 'subject_level_split_analysis' not in self.results:
            ax.text(0.5, 0.5, 'Subject-level split analysis not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        splits = self.results['subject_level_split_analysis']['splits']
        
        train_ages = [self.subject_stats_df[self.subject_stats_df['subject_id'] == sid]['age'].iloc[0] 
                     for sid in splits['train']['subjects']]
        val_ages = [self.subject_stats_df[self.subject_stats_df['subject_id'] == sid]['age'].iloc[0] 
                   for sid in splits['val']['subjects']]
        test_ages = [self.subject_stats_df[self.subject_stats_df['subject_id'] == sid]['age'].iloc[0] 
                    for sid in splits['test']['subjects']]
        
        ax.hist(train_ages, bins=10, alpha=0.5, label='Train', color='blue')
        ax.hist(val_ages, bins=10, alpha=0.5, label='Validation', color='green')
        ax.hist(test_ages, bins=10, alpha=0.8, label='Test', color='red')
        
        ax.set_title('Age Distribution Across Splits', fontweight='bold', fontsize=14)
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('Number of Subjects')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_sample_distribution(self, ax):
        """绘制样本分布"""
        if 'subject_level_split_analysis' not in self.results:
            ax.text(0.5, 0.5, 'Subject-level split analysis not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        stats = self.results['subject_level_split_analysis']['split_stats']
        
        splits = list(stats.keys())
        samples = [stats[split]['n_samples'] for split in splits]
        subjects = [stats[split]['n_subjects'] for split in splits]
        
        x = np.arange(len(splits))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, samples, width, label='Samples', alpha=0.7)
        bars2 = ax.bar(x + width/2, [s*1000 for s in subjects], width, 
                      label='Subjects (×1000)', alpha=0.7)
        
        ax.set_title('Sample and Subject Distribution\nAfter Subject-Level Split', 
                    fontweight='bold', fontsize=14)
        ax.set_xlabel('Data Split')
        ax.set_ylabel('Count')
        ax.set_xticks(x)
        ax.set_xticklabels([s.title() for s in splits])
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 添加数值标签
        for bar in bars1:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{int(height):,}', ha='center', va='bottom', fontsize=10)
    
    def _plot_performance_comparison(self, ax):
        """绘制性能对比"""
        if 'performance_comparison' not in self.results:
            ax.text(0.5, 0.5, 'Performance comparison not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        performances = self.results['performance_comparison']['model_performances']
        
        models = ['Original\n(with leakage)', 'Subject-Level\n(Train-Val)', 'Subject-Level\n(Train-Test)']
        f1_scores = [
            performances['original_split']['f1_macro'],
            performances['subject_level_split']['f1_macro'],
            performances['subject_level_test']['f1_macro']
        ]
        accuracies = [
            performances['original_split']['accuracy'],
            performances['subject_level_split']['accuracy'], 
            performances['subject_level_test']['accuracy']
        ]
        
        x = np.arange(len(models))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, f1_scores, width, label='F1-Score', alpha=0.7, color='skyblue')
        bars2 = ax.bar(x + width/2, accuracies, width, label='Accuracy', alpha=0.7, color='lightcoral')
        
        ax.set_title('Performance Comparison:\nLeakage vs Subject-Level Splits', 
                    fontweight='bold', fontsize=14)
        ax.set_xlabel('Model Type')
        ax.set_ylabel('Performance Score')
        ax.set_xticks(x)
        ax.set_xticklabels(models)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1)
        
        # 添加数值标签
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=10)
    
    def _plot_leakage_impact(self, ax):
        """绘制泄露影响量化"""
        if 'performance_comparison' not in self.results:
            ax.text(0.5, 0.5, 'Performance comparison not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        impact = self.results['performance_comparison']['leakage_impact']
        
        metrics = ['F1-Score\nInflation', 'Accuracy\nInflation', 'F1 Performance\nDrop', 'Accuracy Performance\nDrop']
        values = [
            impact['f1_inflation'] * 100,
            impact['accuracy_inflation'] * 100,
            impact['performance_drop_f1'] * 100,
            impact['performance_drop_acc'] * 100
        ]
        colors = ['red', 'red', 'orange', 'orange']
        
        bars = ax.bar(metrics, values, color=colors, alpha=0.7)
        ax.set_title('Data Leakage Impact Quantification', fontweight='bold', fontsize=14)
        ax.set_ylabel('Percentage (%)')
        ax.grid(True, alpha=0.3)
        
        # 添加数值标签
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                   f'{value:.1f}%', ha='center', va='bottom', fontsize=10)
        
        # 添加零线
        ax.axhline(0, color='black', linestyle='-', alpha=0.3)
    
    def _plot_class_distribution(self, ax):
        """绘制类别分布对比"""
        if 'subject_level_split_analysis' not in self.results:
            ax.text(0.5, 0.5, 'Subject-level split analysis not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        splits = self.results['subject_level_split_analysis']['splits']
        
        # 计算类别分布
        train_dist = np.bincount(splits['train']['y'], minlength=self.original_data['n_classes'])
        val_dist = np.bincount(splits['val']['y'], minlength=self.original_data['n_classes'])
        test_dist = np.bincount(splits['test']['y'], minlength=self.original_data['n_classes'])
        
        # 归一化为百分比
        train_pct = train_dist / train_dist.sum() * 100
        val_pct = val_dist / val_dist.sum() * 100
        test_pct = test_dist / test_dist.sum() * 100
        
        # 只显示前10个类别（避免过于拥挤）
        n_show = min(10, len(train_pct))
        x = np.arange(n_show)
        width = 0.25
        
        ax.bar(x - width, train_pct[:n_show], width, label='Train', alpha=0.7)
        ax.bar(x, val_pct[:n_show], width, label='Validation', alpha=0.7)
        ax.bar(x + width, test_pct[:n_show], width, label='Test', alpha=0.7)
        
        ax.set_title('Class Distribution Across Splits\n(Top 10 Classes)', 
                    fontweight='bold', fontsize=14)
        ax.set_xlabel('Brain Region Class')
        ax.set_ylabel('Percentage (%)')
        ax.set_xticks(x)
        ax.set_xticklabels([f'C{i}' for i in range(n_show)])
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    def _plot_performance_waterfall(self, ax):
        """绘制性能下降瀑布图"""
        if 'performance_comparison' not in self.results:
            ax.text(0.5, 0.5, 'Performance comparison not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        performances = self.results['performance_comparison']['model_performances']
        
        # F1得分的变化
        original_f1 = performances['original_split']['f1_macro']
        subject_val_f1 = performances['subject_level_split']['f1_macro']
        subject_test_f1 = performances['subject_level_test']['f1_macro']
        
        steps = ['Original\n(with leakage)', 'Remove\nleakage', 'True\ngeneralization']
        values = [original_f1, subject_val_f1, subject_test_f1]
        changes = [0, subject_val_f1 - original_f1, subject_test_f1 - subject_val_f1]
        
        colors = ['blue', 'red', 'red']
        
        # 绘制瀑布图
        ax.bar(steps[0], values[0], color=colors[0], alpha=0.7)
        for i in range(1, len(steps)):
            ax.bar(steps[i], values[i], color=colors[i], alpha=0.7)
            
            # 绘制连接线和变化标注
            if i > 0:
                ax.annotate(f'{changes[i]:.3f}', 
                           xy=(i-0.5, (values[i-1] + values[i])/2),
                           xytext=(i-0.5, (values[i-1] + values[i])/2 + 0.02),
                           ha='center', va='bottom', fontsize=10,
                           arrowprops=dict(arrowstyle='->', color='black', alpha=0.5))
        
        ax.set_title('Performance Waterfall:\nFrom Leakage to True Generalization', 
                    fontweight='bold', fontsize=14)
        ax.set_ylabel('F1-Score')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, max(values) * 1.1)
    
    def _plot_age_vs_samples(self, ax):
        """绘制年龄vs样本数散点图"""
        df = self.subject_stats_df
        
        scatter = ax.scatter(df['age'], df['n_samples'], 
                           c=df['subject_id'], cmap='viridis', 
                           alpha=0.7, s=60)
        
        # 标记测试受试者
        test_row = df[df['subject_id'] == 38]
        if len(test_row) > 0:
            ax.scatter(test_row['age'], test_row['n_samples'], 
                      c='red', s=100, marker='*', label='Test Subject')
        
        ax.set_title('Age vs Sample Count by Subject', fontweight='bold', fontsize=14)
        ax.set_xlabel('Age (years)')
        ax.set_ylabel('Number of Samples')
        ax.grid(True, alpha=0.3)
        
        if len(test_row) > 0:
            ax.legend()
        
        # 添加回归线
        z = np.polyfit(df['age'], df['n_samples'], 1)
        p = np.poly1d(z)
        ax.plot(df['age'], p(df['age']), "r--", alpha=0.8, linewidth=2)
    
    def _plot_training_size_impact(self, ax):
        """绘制训练集规模影响"""
        if 'subject_level_split_analysis' not in self.results:
            ax.text(0.5, 0.5, 'Subject-level split analysis not available', 
                   ha='center', va='center', transform=ax.transAxes)
            return
            
        stats = self.results['subject_level_split_analysis']['split_stats']
        
        # 模拟不同训练集大小的影响
        train_subjects = len(stats['train']['n_subjects'])
        total_subjects = train_subjects + len(stats['val']['n_subjects'])
        
        # 创建假设的学习曲线
        subject_ratios = np.linspace(0.1, 1.0, 10)
        n_subjects_used = subject_ratios * train_subjects
        
        # 简单的学习曲线模型（基于经验公式）
        estimated_performance = 1 - np.exp(-n_subjects_used / 10) * 0.3
        
        ax.plot(n_subjects_used, estimated_performance, 'o-', linewidth=2, markersize=6)
        ax.axvline(train_subjects, color='red', linestyle='--', 
                  label=f'Current: {train_subjects} subjects')
        
        ax.set_title('Estimated Training Size Impact', fontweight='bold', fontsize=14)
        ax.set_xlabel('Number of Training Subjects')
        ax.set_ylabel('Estimated Performance')
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    def _plot_confusion_matrix_comparison(self, ax):
        """绘制混淆矩阵对比（简化版）"""
        ax.text(0.5, 0.5, 'Confusion Matrix Comparison\n(Requires model predictions)', 
               ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title('Confusion Matrix Comparison', fontweight='bold', fontsize=14)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    
    def _plot_summary_dashboard(self, ax):
        """绘制总结仪表盘"""
        # 清除坐标轴
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        
        # 添加总结信息
        summary_text = "SUMMARY DASHBOARD\n\n"
        
        if 'original_split_analysis' in self.results:
            leakage = self.results['original_split_analysis']['leakage_metrics']
            summary_text += f"Data Leakage Severity: {leakage['overlap_ratio']*100:.1f}%\n"
        
        if 'performance_comparison' in self.results:
            impact = self.results['performance_comparison']['leakage_impact']
            summary_text += f"Performance Inflation: {impact['f1_inflation']*100:.1f}%\n"
            summary_text += f"Estimated True F1: {impact['estimated_true_f1']:.3f}\n"
        
        summary_text += f"\nRecommendations:\n"
        summary_text += f"• Use subject-level splits\n"
        summary_text += f"• Implement age correction\n"
        summary_text += f"• Consider subject embedding"
        
        ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, 
               fontsize=12, va='top', ha='left',
               bbox=dict(boxstyle="round,pad=0.5", facecolor="lightblue", alpha=0.8))
        
        ax.set_title('Analysis Summary', fontweight='bold', fontsize=16)
    
    def _generate_individual_plots(self):
        """生成单独的高质量图表"""
        print("🔄 正在生成单独的高质量图表...")
        
        # 1. 数据泄露诊断图
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        self._plot_leakage_visualization(ax)
        plt.tight_layout()
        plt.savefig(os.path.join(self.save_path, 'visualizations', 'data_leakage_diagnosis.png'), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        # 2. 性能对比图
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        self._plot_performance_comparison(ax)
        plt.tight_layout()
        plt.savefig(os.path.join(self.save_path, 'visualizations', 'performance_comparison.png'), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        print("✅ 单独图表生成完成")
    
    def generate_comprehensive_report(self):
        """
        生成完整的分析报告
        """
        print("\n" + "="*80)
        print("📝 Phase 6: 生成综合分析报告")
        print("="*80)
        
        report_path = os.path.join(self.save_path, 'reports', 'data_leakage_analysis_report.md')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("# 脑影像数据泄露诊断与修复分析报告\n\n")
            f.write(f"**生成时间**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 执行摘要
            f.write("## 执行摘要\n\n")
            self._write_executive_summary(f)
            
            # 数据概况
            f.write("## 数据概况\n\n")
            self._write_data_overview(f)
            
            # 泄露诊断结果
            f.write("## 数据泄露诊断结果\n\n")
            self._write_leakage_diagnosis(f)
            
            # 性能影响分析
            f.write("## 性能影响分析\n\n")
            self._write_performance_impact(f)
            
            # 修复建议
            f.write("## 修复建议与下一步行动\n\n")
            self._write_recommendations(f)
            
            # 技术细节
            f.write("## 技术实施细节\n\n")
            self._write_technical_details(f)
        
        # 保存结果到JSON
        results_path = os.path.join(self.save_path, 'reports', 'analysis_results.json')
        with open(results_path, 'w', encoding='utf-8') as f:
            # 转换numpy数组为列表以便JSON序列化
            json_results = self._prepare_results_for_json()
            json.dump(json_results, f, indent=2, ensure_ascii=False)
        
        print(f"✅ 综合分析报告已保存:")
        print(f"  - Markdown报告: {report_path}")
        print(f"  - JSON结果: {results_path}")
        
        return report_path, results_path
    
    def _write_executive_summary(self, f):
        """写入执行摘要"""
        if 'original_split_analysis' in self.results and 'performance_comparison' in self.results:
            leakage = self.results['original_split_analysis']['leakage_metrics']
            impact = self.results['performance_comparison']['leakage_impact']
            
            f.write(f"本次分析发现了严重的数据泄露问题：\n\n")
            f.write(f"- **受试者重叠率**: {leakage['overlap_ratio']*100:.1f}%\n")
            f.write(f"- **性能虚高程度**: F1得分虚高{impact['f1_inflation']*100:.1f}%\n")
            f.write(f"- **预估真实性能**: F1={impact['estimated_true_f1']:.3f}\n")
            f.write(f"- **性能下降幅度**: {impact['performance_drop_f1']:.3f}\n\n")
            
            if impact['f1_inflation'] > 0.3:
                f.write("**严重性评估**: 数据泄露导致的性能虚高超过30%，必须立即修复。\n\n")
            elif impact['f1_inflation'] > 0.1:
                f.write("**严重性评估**: 数据泄露导致显著的性能虚高，建议优先修复。\n\n")
        else:
            f.write("执行摘要待完善（缺少分析结果）。\n\n")
    
    def _write_data_overview(self, f):
        """写入数据概况"""
        f.write(f"- **总受试者数**: {self.original_data['n_subjects']}\n")
        f.write(f"- **总样本数**: {self.original_data['n_samples']:,}\n")
        f.write(f"- **特征维度**: {self.original_data['X'].shape[1]}\n")
        f.write(f"- **分类任务**: {self.original_data['n_classes']} 个脑区\n")
        f.write(f"- **年龄范围**: {self.subject_stats_df['age'].min():.1f} - {self.subject_stats_df['age'].max():.1f} 岁\n")
        f.write(f"- **平均每受试者样本数**: {self.subject_stats_df['n_samples'].mean():.0f}\n\n")
    
    def _write_leakage_diagnosis(self, f):
        """写入泄露诊断结果"""
        if 'original_split_analysis' in self.results:
            analysis = self.results['original_split_analysis']
            metrics = analysis['leakage_metrics']
            severity = analysis['leakage_severity']
            
            f.write(f"### 泄露程度量化\n\n")
            f.write(f"- **训练集涉及受试者**: {metrics['train_subjects']} 个\n")
            f.write(f"- **验证集涉及受试者**: {metrics['val_subjects']} 个\n")
            f.write(f"- **重叠受试者数量**: {metrics['overlapping_subjects']} 个\n")
            f.write(f"- **受试者重叠率**: {metrics['overlap_ratio']*100:.1f}%\n")
            f.write(f"- **泄露严重程度**: {severity['severity_level']} (得分: {severity['severity_score']:.3f})\n\n")
            
            f.write(f"### 样本级别泄露分析\n\n")
            f.write(f"- **训练集泄露样本比例**: {severity['train_samples_leaked_ratio']*100:.1f}%\n")
            f.write(f"- **验证集泄露样本比例**: {severity['val_samples_leaked_ratio']*100:.1f}%\n\n")
        else:
            f.write("泄露诊断结果待完善。\n\n")
    
    def _write_performance_impact(self, f):
        """写入性能影响分析"""
        if 'performance_comparison' in self.results:
            performances = self.results['performance_comparison']['model_performances']
            impact = self.results['performance_comparison']['leakage_impact']
            
            f.write(f"### 性能对比结果\n\n")
            f.write(f"| 分割方法 | F1-Score | Accuracy |\n")
            f.write(f"|---------|----------|----------|\n")
            f.write(f"| 原始分割 (带泄露) | {performances['original_split']['f1_macro']:.4f} | {performances['original_split']['accuracy']:.4f} |\n")
            f.write(f"| 受试者级别 (训练-验证) | {performances['subject_level_split']['f1_macro']:.4f} | {performances['subject_level_split']['accuracy']:.4f} |\n")
            f.write(f"| 受试者级别 (训练-测试) | {performances['subject_level_test']['f1_macro']:.4f} | {performances['subject_level_test']['accuracy']:.4f} |\n\n")
            
            f.write(f"### 泄露影响量化\n\n")
            f.write(f"- **F1性能虚高**: {impact['f1_inflation']*100:.1f}%\n")
            f.write(f"- **准确率虚高**: {impact['accuracy_inflation']*100:.1f}%\n")
            f.write(f"- **F1性能下降**: {impact['performance_drop_f1']:.3f}\n")
            f.write(f"- **准确率下降**: {impact['performance_drop_acc']:.3f}\n\n")
        else:
            f.write("性能影响分析待完善。\n\n")
    
    def _write_recommendations(self, f):
        """写入修复建议"""
        f.write(f"### 立即行动项\n\n")
        f.write(f"1. **数据重新分割**: 严格按受试者ID进行训练/验证/测试分割\n")
        f.write(f"2. **性能重新评估**: 使用新分割重新训练模型并建立真实baseline\n")
        f.write(f"3. **年龄效应分析**: 检查训练集与测试受试者的年龄差异\n")
        f.write(f"4. **Subject Embedding实施**: 设计受试者嵌入层解决个体差异\n\n")
        
        f.write(f"### 中期优化策略\n\n")
        f.write(f"1. **年龄校正**: 实施年龄回归校正或协变量控制\n")
        f.write(f"2. **数据增强**: 考虑跨受试者的数据增强方法\n")
        f.write(f"3. **集成方法**: 结合多种泛化技术\n")
        f.write(f"4. **交叉验证**: 实施Leave-One-Subject-Out交叉验证\n\n")
        
        f.write(f"### 长期研究方向\n\n")
        f.write(f"1. **多中心验证**: 收集更多测试受试者数据\n")
        f.write(f"2. **个性化模型**: 开发自适应个体化分类器\n")
        f.write(f"3. **领域自适应**: 研究更先进的域适应方法\n\n")
    
    def _write_technical_details(self, f):
        """写入技术实施细节"""
        f.write(f"### 数据分割策略\n\n")
        f.write(f"```python\n")
        f.write(f"# 推荐的受试者级别分割代码\n")
        f.write(f"train_subjects = [1, 2, ..., 30]  # 30个受试者训练\n")
        f.write(f"val_subjects = [31, 32, ..., 37]  # 7个受试者验证\n")
        f.write(f"test_subjects = [38]              # 1个受试者测试\n")
        f.write(f"```\n\n")
        
        f.write(f"### 性能评估协议\n\n")
        f.write(f"1. **交叉验证**: 使用GroupKFold确保受试者不重叠\n")
        f.write(f"2. **评估指标**: 优先使用macro-averaged F1和balanced accuracy\n")
        f.write(f"3. **置信区间**: 报告性能指标的95%置信区间\n")
        f.write(f"4. **统计检验**: 使用配对t检验比较不同方法\n\n")
        
        f.write(f"### Subject Embedding架构建议\n\n")
        f.write(f"```python\n")
        f.write(f"# 推荐的双通道架构\n")
        f.write(f"subject_embedding_dim = 64\n")
        f.write(f"age_embedding_dim = 16\n")
        f.write(f"combined_features = concatenate([brain_features, subject_emb, age_emb])\n")
        f.write(f"```\n\n")
    
    def _prepare_results_for_json(self):
        """准备结果用于JSON序列化"""
        json_results = {}
        
        for key, value in self.results.items():
            json_results[key] = self._convert_to_serializable(value)
            
        return json_results
    
    def _convert_to_serializable(self, obj):
        """递归转换对象为JSON可序列化格式"""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, dict):
            return {k: self._convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._convert_to_serializable(item) for item in obj]
        else:
            return obj
    
    def run_complete_analysis(self, X_train_current=None, X_val_current=None, 
                            y_train_current=None, y_val_current=None):
        """
        运行完整的数据泄露分析流程
        
        Args:
            X_train_current: 当前使用的训练集特征
            X_val_current: 当前使用的验证集特征
            y_train_current: 当前使用的训练集标签
            y_val_current: 当前使用的验证集标签
        
        Returns:
            dict: 包含所有分析结果和新数据分割的字典
        """
        
        print("🚀 开始完整的数据泄露诊断与修复分析")
        print("="*80)
        start_time = time.time()
        
        # Phase 1: 加载数据
        self.load_data_with_metadata()
        
        # Phase 2: 分析当前分割的泄露程度
        if all(x is not None for x in [X_train_current, X_val_current, y_train_current, y_val_current]):
            print("\n🔍 分析当前数据分割...")
            self.analyze_original_split_leakage(X_train_current, X_val_current, 
                                              y_train_current, y_val_current)
        else:
            print("\n⚠️ 未提供当前分割数据，跳过泄露分析")
        
        # Phase 3: 创建新的受试者级别分割
        print("\n🔄 创建受试者级别分割...")
        new_splits = self.create_subject_level_splits()
        
        # Phase 4: 性能对比分析
        if all(x is not None for x in [X_train_current, X_val_current, y_train_current, y_val_current]):
            print("\n⚖️ 进行性能对比分析...")
            self.evaluate_performance_comparison(X_train_current, X_val_current, 
                                                y_train_current, y_val_current)
        else:
            print("\n⚠️ 无法进行性能对比（缺少原始分割数据）")
        
        # Phase 5: 生成可视化
        print("\n📊 生成可视化分析...")
        self.generate_comprehensive_visualizations()
        
        # Phase 6: 生成报告
        print("\n📝 生成分析报告...")
        report_path, results_path = self.generate_comprehensive_report()
        
        # 计算运行时间
        end_time = time.time()
        runtime = end_time - start_time
        
        print(f"\n🎉 完整分析已完成!")
        print(f"📁 结果保存在: {self.save_path}")
        print(f"⏱️ 总运行时间: {runtime:.2f} 秒")
        print(f"📄 报告路径: {report_path}")
        
        # 返回完整结果
        return {
            'analysis_results': self.results,
            'new_data_splits': new_splits,
            'subject_stats': self.subject_stats_df,
            'runtime': runtime,
            'save_path': self.save_path,
            'report_path': report_path,
            'results_path': results_path
        }


# ============================================================================
# 🚀 使用示例和主函数
# ============================================================================

def main_analysis_pipeline(data_path, current_splits=None, save_path='./data_leakage_analysis/'):
    """
    主分析流程
    
    Args:
        data_path: TRAIN38.mat文件路径
        current_splits: 当前分割数据的字典，包含X_train, X_val, y_train, y_val
        save_path: 结果保存路径
    
    Returns:
        完整的分析结果
    """
    
    print("🧠 脑影像数据泄露诊断与修复分析")
    print("="*80)
    print(f"📂 数据路径: {data_path}")
    print(f"💾 保存路径: {save_path}")
    
    # 初始化分析器
    analyzer = DataLeakageAnalyzer(data_path=data_path, save_path=save_path)
    
    # 运行完整分析
    if current_splits is not None:
        results = analyzer.run_complete_analysis(
            X_train_current=current_splits.get('X_train'),
            X_val_current=current_splits.get('X_val'),
            y_train_current=current_splits.get('y_train'),
            y_val_current=current_splits.get('y_val')
        )
    else:
        results = analyzer.run_complete_analysis()
    
    return results, analyzer


# ============================================================================
# 💡 实际使用代码示例
# ============================================================================

if __name__ == "__main__":
    # 设置数据路径
    DATA_PATH = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'
    SAVE_PATH = './data_leakage_analysis_results/'
    
    # 如果你有当前的分割数据，可以这样提供：
    current_splits = {
        'X_train': X_train_scaled,    # 你当前的训练数据
        'X_val': X_val_scaled,        # 你当前的验证数据
        'y_train': y_train,           # 你当前的训练标签
        'y_val': y_val                # 你当前的验证标签
    }
    
    # 运行完整分析
    print("🚀 开始数据泄露分析...")
    results, analyzer = main_analysis_pipeline(
        data_path=DATA_PATH,
        current_splits=current_splits,  # 如果没有当前分割数据，可以设为None
        save_path=SAVE_PATH
    )
    
    # 获取新的数据分割
    new_splits = results['new_data_splits']
    
    print("\n✅ 分析完成！新的数据分割已准备就绪：")
    print(f"📊 训练集: {new_splits['train']['X'].shape}")
    print(f"📊 验证集: {new_splits['val']['X'].shape}")
    print(f"📊 测试集: {new_splits['test']['X'].shape}")
    
    # 可以直接使用新的分割数据
    X_train_new = new_splits['train']['X']
    y_train_new = new_splits['train']['y']
    X_val_new = new_splits['val']['X']
    y_val_new = new_splits['val']['y']
    X_test_new = new_splits['test']['X']
    y_test_new = new_splits['test']['y']
    
    # 获取受试者信息（用于后续的Subject Embedding）
    train_subject_ids = new_splits['train']['subject_ids']
    val_subject_ids = new_splits['val']['subject_ids']
    test_subject_ids = new_splits['test']['subject_ids']
    
    # 获取年龄信息
    train_ages = new_splits['train']['ages']
    val_ages = new_splits['val']['ages']
    test_ages = new_splits['test']['ages']
    
    print("\n🎯 下一步建议：")
    print("1. 使用新的数据分割重新训练你的模型")
    print("2. 对比新旧性能，量化泄露影响")
    print("3. 分析年龄效应对泛化的影响")
    print("4. 设计Subject Embedding解决个体差异")
    
    # 保存新分割的数据（可选）
    import pickle
    splits_save_path = os.path.join(SAVE_PATH, 'new_data_splits.pkl')
    with open(splits_save_path, 'wb') as f:
        pickle.dump(new_splits, f)
    print(f"\n💾 新数据分割已保存到: {splits_save_path}")


# ============================================================================
# 🔧 便捷函数：快速获取修复后的数据
# ============================================================================

def get_fixed_data_splits(data_path, save_path='./temp_analysis/'):
    """
    快速获取修复后的数据分割（无需分析当前数据）
    
    Args:
        data_path: TRAIN38.mat文件路径
        save_path: 临时保存路径
    
    Returns:
        修复后的数据分割字典
    """
    analyzer = DataLeakageAnalyzer(data_path=data_path, save_path=save_path)
    analyzer.load_data_with_metadata()
    new_splits = analyzer.create_subject_level_splits()
    
    return new_splits

def compare_with_current_splits(data_path, X_train_current, X_val_current, 
                              y_train_current, y_val_current, 
                              save_path='./leakage_comparison/'):
    """
    专门用于对比当前分割与修复后分割的性能差异
    
    Args:
        data_path: TRAIN38.mat文件路径
        X_train_current, X_val_current: 当前的训练/验证特征
        y_train_current, y_val_current: 当前的训练/验证标签
        save_path: 结果保存路径
    
    Returns:
        性能对比结果
    """
    analyzer = DataLeakageAnalyzer(data_path=data_path, save_path=save_path)
    
    # 运行对比分析
    results = analyzer.run_complete_analysis(
        X_train_current=X_train_current,
        X_val_current=X_val_current,
        y_train_current=y_train_current,
        y_val_current=y_val_current
    )
    
    return results['analysis_results']['performance_comparison']